# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client:
 Version:    29.8.1
 Context:    default
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.141.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /usr/local/lib/docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /usr/local/lib/docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /usr/local/lib/docker/cli-plugins/docker-dh

### What is a container?

A container is an isolated environment in which software can run, together with its required dependencies. It is created based on its container image.

Containers can be compared to sandboxes because they isolate applications and their dependencies from one eachother and from the host system. However, unlike virtual machines, containers share the host operating system's kernel rather than requiring a separate operating system.

(https://www.docker.com/resources/what-container/)

### Why do we use containers?

Different tools often require different software versions or dependencies. This can cause compatibility problems. Using different containers for each tool solves that. Another advantage is that we dont have to install and configure every tool manually.
Also it improves reproducibility since researchers can use the same software enviroment. Another advantage is portability, since containers can run on different computers without having to reinstall the software and its dependencies, as long as the required container technology is supported.


### What is a docker image?

A Docker container image is like a template which defines everything needed to run an application: code, runtime, system tools, system libraries and settings. So Docker creates containers based on the containers image. 

### Let's run our first docker image:

### Login to docker

In [2]:
# This you need to do on the command line directly

### Run your first docker container

In [3]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [ ]:
"docker ps" finds all running containers
"docker ps -a" finds all running containers including stopped ones

CONTAINER ID   IMAGE                                         COMMAND                  CREATED          STATUS                      PORTS     NAMES
fc191aa7a3cc   hello-world                                   "/hello"                 27 minutes ago   Exited (0) 27 minutes ago             elegant_shaw

### Delete the container again, give prove its deleted

In [ ]:
delete a specific container with: docker rm <container_id>

Or:
docker run --rm removes the container automatically after running, nextflow delets the used containers automaticaly

In [ ]:
(computational-workflows) laura@DESKTOP-5NCS9RT:~/computational-workflows-2026$ docker ps -a
CONTAINER ID   IMAGE                                         COMMAND                  CREATED        STATUS                      PORTS     NAMES
d5fa56c5c851   quay.io/biocontainers/fq:0.12.0--h9ee0642_0   "/usr/local/env-exec…"   43 hours ago   Exited (137) 43 hours ago             nxf-lelH5bjQsVWqWoAiTAxO7W1o

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. first create a diretcory:
mkdir -p fastqc_manual
cd fastqc_manual

download FastQC from website

copie the download:
cp /mnt/c/Users/Laura/Downloads/fastqc_v0.12.1.zip .

unzip the folder:
 unzip fastqc_v0.12.1.zip

make it executable:
chmod +x FastQC/fastqc

Cornfirm the installation:
./FastQC/fastqc --version

2. create output directory mkdir -p day_04/fastqc_results
3. run FastQC:
./FastQC/fastqc \
  ../../day_02/fetchngs_out/fastq/SRX19144488_SRR23195511_1.fastq.gz \
  -o ../fastqc_results


### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

In [ ]:
# run the container and save the results to a new "fastqc_results" directory
docker run --rm \
  -v "$(pwd)/../day_02/fetchngs_out/fastq:/data:ro" \
  -v "$(pwd)/fastqc_docker_results:/results" \
  community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
  fastqc /data/SRX19144488_SRR23195511_1.fastq.gz -o /results

# we mount the neede files so docker has access to them, docker is like a different 
# machine so it doesnt has access to our file sjust like that. -v is like a mapping 

# with just bash after docker run..... we get in an interactive version? 
# There we can see that we can change the files inside our docker container if we mounted it first
# in example he mounted a complete directory? 

example test run:
docker run --rm -it community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae FastQC -h

docker goes inside the container and runs FastQC -h and the exits again.

warning: downloada image missmatch betwene platform and software -> docker is able to resolve this and tsill runs


### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

Eventough the manual installation was straight forward it included way more steps than using docker. So I would say using docker was easier and likely will be easier in the future since we can reuse the image.

### What would you say, which approach is more reproducible?

Using docker is more reproducible since we can use the same image across different computers.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

Eventhough it is a bit hard to make out the correct fastq file in the multiQC report the results seem to be identical.
The MultiQC report from day 2 confirms that we used the same FastQC version (0.12.1) as we used today for both FastQC runs.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor
!code my_dockerfile

### Explain the RUN and ENV lines you added to the file

RUN apt-get update && apt-get install -y curl cowsay

- RUN <command> - this instruction tells the builder to run the specified command.
- apt-get update: Updates Debian's list of available software packages.
- apt-get install -y cowsay: Installs cowsay and automatically confirms the installation.
- -y: answer yes automaticaly to confirm prompts during update and isntallation
More Info: https://manpages.debian.org/trixie/apt/apt-get.8.en.html
and https://docs.docker.com/reference/dockerfile

ENV PATH="/usr/games:${PATH}"
Defines an environment variable inside the image. We used it to add /usr/games to PATH, allowing us to execute cowsay without specifying its full location.

In [ ]:
# build the docker image
docker build -f my_dockerfile -t my-cowsay .
docker build --pull --no-cache -f my_dockerfile -t my-cowsay .

-f my_dockerfile: Specifies the Dockerfile.
-t my-cowsay: Gives the new image a name.
.: Uses the current directory as the build context.

In [ ]:
# make sure that the image has been built
docker images

# just look for my-cowsay name in the list -> it was there

In [ ]:
# run the docker file 
docker run --rm my-cowsay cowsay "Hello!"

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [10]:
# use the file "salmon_docker" in this directory to build a new docker image

In [ ]:
# build the image
!docker build -f salmon_docker -t my-salmon .

In [14]:
# run the docker image to give out the version of salmon
!docker run --rm my-salmon salmon --version

# salmon 1.5.2

salmon 1.5.2


## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

What is https://biocontainers.pro/ ?

No, we don't need to create a Docker image every time. We can search for existing images in repositories such as BioContainers or use Seqera Containers. If no image with the required software versions and dependencies exists,  we would need to build our own image. 

docker pull quay.io/biocontainers/salmon:2.8.0--hfa8f182_0 (command from website, tag for the latest version on website found under "tags")

Get the version:
docker run quay.io/biocontainers/salmon:2.8.0--hfa8f182_0 salmon --version
Out:
salmon 2.8.0

BioContainers is a community-driven project for creating, distributing and maintaining containers for scientific software. According to the project, it provides more than 100,000 containers covering over 12,000 tools. It simplifies software installation and dependency management by providing ready-to-use container images. These images can be used across different computing environments, improving portability and reproducibility.

## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

Yes, there are other ways. If a suitable image is not already available, for example on BioContainers, we can use Seqera Containers to generate one on demand. Seqera Containers is a free, web-based service that generates container images from software packages.
We only need to select the required software packages and versions, and Seqera Containers automatically generates a container image without requiring us to write a Dockerfile. Seqera Containers creates a uique hash based on the requested packages and version, identical requests would produce the same hash and they reuse the exact same stored container image, which improves reproducibility